# Rail Corrugation
## Feature Experiments

This notebook evaluates improvements to the baseline feature representation.

Experiments:
1. Baseline vibration features
2. Improved PSD aggregation
3. Shock features
4. Combined feature set

All experiments use the same stratified cross-validation procedure and Macro F1 metric.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import welch
from scipy.stats import kurtosis, skew

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

DATA_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "Train"

FS = 10000

In [2]:
baseline = pd.read_csv(DATA_DIR / "rail_features.csv")

labels = baseline[["filename", "label"]]

X_base = baseline.drop(columns=["filename", "label"])
y = baseline["label"]

In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

In [4]:
# experiment a
base_scores = cross_val_score(
    rf,
    X_base,
    y,
    cv=cv,
    scoring="f1_macro"
)

print(base_scores.round(3))
print("Baseline:", base_scores.mean().round(3))

[0.725 0.652 0.779 0.756 0.666]
Baseline: 0.716


## Experiment B — PSD Aggregation

The baseline averages axle-box signals before calculating PSD.

Because different axle boxes may have different signal phases, this can cause cancellation.

The alternative approach calculates PSD independently for each axle box before averaging spectral power across the rail side.

In [5]:
sample = pd.read_csv(TRAIN_PATH / "Train1.csv")

vib_cols = [c for c in sample.columns if "vibration" in c.lower()]

side1_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [1, 3, 5, 7])]
side2_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [2, 4, 6, 8])]

In [6]:
bands = [
    (0, 100),
    (100, 250),
    (250, 500),
    (500, 1000),
    (1000, 2000),
    (2000, 5000)
]

In [7]:
def mean_psd(data, columns):
    psds = []

    for col in columns:
        f, psd = welch(data[col].to_numpy(), fs=FS, nperseg=2048)
        psds.append(psd)

    return f, np.mean(psds, axis=0)

In [8]:
def psd_features(data, columns):
    f, psd = mean_psd(data, columns)

    features = {}

    for low, high in bands:
        mask = (f >= low) & (f < high)
        features[f"power_{low}_{high}"] = np.trapezoid(psd[mask], f[mask])

    return features

In [9]:
rows = []

for _, row in labels.iterrows():
    data = pd.read_csv(TRAIN_PATH / row["filename"])

    s1 = psd_features(data, side1_vib)
    s2 = psd_features(data, side2_vib)

    features = {}

    features.update({f"new_side1_{k}": v for k, v in s1.items()})
    features.update({f"new_side2_{k}": v for k, v in s2.items()})

    rows.append(features)

psd_df = pd.DataFrame(rows)

In [10]:
X_psd = pd.concat(
    [X_base.reset_index(drop=True), psd_df],
    axis=1
)

In [11]:
psd_scores = cross_val_score(
    rf,
    X_psd,
    y,
    cv=cv,
    scoring="f1_macro"
)

print(psd_scores.round(3))
print("Improved PSD:", psd_scores.mean().round(3))

[0.609 0.639 0.723 0.73  0.666]
Improved PSD: 0.673


## Experiment C — Shock Features

The baseline uses only vibration measurements.

Shock signals may provide complementary information about impulsive wheel-rail interactions associated with corrugation.

In [12]:
shock_cols = [c for c in sample.columns if "shock" in c.lower()]

side1_shock = [c for c in shock_cols if any(f"position {p}" in c.lower() for p in [1, 3, 5, 7])]
side2_shock = [c for c in shock_cols if any(f"position {p}" in c.lower() for p in [2, 4, 6, 8])]

In [13]:
def shock_features(data, columns):
    x = data[columns].to_numpy().ravel()

    return {
        "rms": np.sqrt(np.mean(x ** 2)),
        "std": np.std(x),
        "peak": np.max(np.abs(x)),
        "kurtosis": kurtosis(x)
    }

In [14]:
rows = []

for _, row in labels.iterrows():
    data = pd.read_csv(TRAIN_PATH / row["filename"])

    s1 = shock_features(data, side1_shock)
    s2 = shock_features(data, side2_shock)

    features = {}

    features.update({f"shock_side1_{k}": v for k, v in s1.items()})
    features.update({f"shock_side2_{k}": v for k, v in s2.items()})

    features["shock_rms_diff"] = s1["rms"] - s2["rms"]
    features["shock_rms_ratio"] = s1["rms"] / s2["rms"]

    rows.append(features)

shock_df = pd.DataFrame(rows)

In [15]:
X_shock = pd.concat(
    [X_base.reset_index(drop=True), shock_df],
    axis=1
)

shock_scores = cross_val_score(
    rf,
    X_shock,
    y,
    cv=cv,
    scoring="f1_macro"
)

print(shock_scores.round(3))
print("Shock:", shock_scores.mean().round(3))

[0.729 0.652 0.848 0.649 0.481]
Shock: 0.672


In [16]:
X_combined = pd.concat(
    [
        X_base.reset_index(drop=True),
        psd_df,
        shock_df
    ],
    axis=1
)

combined_scores = cross_val_score(
    rf,
    X_combined,
    y,
    cv=cv,
    scoring="f1_macro"
)

print(combined_scores.round(3))
print("Combined:", combined_scores.mean().round(3))

[0.725 0.695 0.723 0.645 0.652]
Combined: 0.688


In [17]:
results = pd.DataFrame({
    "Baseline": base_scores,
    "Improved PSD": psd_scores,
    "Shock": shock_scores,
    "Combined": combined_scores
})

In [18]:
summary = pd.DataFrame({
    "Mean Macro F1": results.mean(),
    "Std": results.std()
})

display(summary)

,Mean Macro F1,Std
Baseline,0.715646,0.055380
Improved PSD,0.673393,0.052304
Shock,0.671881,0.133928
Combined,0.687924,0.037811


In [19]:
display(results)

,Baseline,Improved PSD,Shock,Combined
0,0.724755,0.609278,0.728723,0.724755
1,0.652482,0.639486,0.652482,0.695042
2,0.779487,0.722581,0.848268,0.722581
3,0.755871,0.729988,0.649123,0.645390
4,0.665633,0.665633,0.480808,0.651852
